# Music Annotation Project V9.0B
## Notebook 60R2: Fresh SAFE_REMOVE inference on reviewed HITL masks

**Runtime:** CPU. In Google Colab choose **Runtime -> Run all**.

60R2 revises 60R1 to consume the Notebook 59R1 deployment bundle. The `.joblib` files are loaded from the frozen Drive bundle and are not embedded in this notebook.

### Fixed rules
- Notebook 07 aligned observed page and reviewed/corrected mask are the page authority.
- No clean page, training, fitting, recalibration, threshold movement or SAFE_PROTECT.
- Construct the exact 17 predictors in `predictor_schema.json` order.
- Frozen REMOVE threshold 0.995, page-global 8-connectivity, minimum run 2, region mean 0.995, every region pixel 0.995.
- `M5_REMOVE = M3 & ~SAFE_REMOVE`; all unsupported/invalid/failed coordinates retain exact M3.
- No pixel outside the prepared authoritative mask may change.
- Research/HITL evidence only. Production authority remains false.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-learn joblib matplotlib


## 1. Configuration and helpers


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, sys, platform, time, shutil, traceback, zipfile
import cv2, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
N59_ROOT=ROOT/'observable_residual_mask_m5_v8_8_deployment_freeze'
NOTEBOOK59_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'v9_0b_notebook60r2_fresh_safe_remove'
PAGE_ALLOWLIST=None; EXPECTED_REVIEWED_PAGES=151; REQUIRE_EXACT_151=False; STOP_AT_FIRST_PAGE_FAILURE=False; MAX_VISUAL_PAGES=30
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook60r2_v9_0b'; OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; FAILURES=OUT/'failures'; VISUAL=OUT/'visual'; UPLOAD=OUT/'upload'
for d in [OUT,INPUTS,PAGES,TABLES,AUDIT,FAILURES,VISUAL,UPLOAD]: d.mkdir(parents=True,exist_ok=False)
def require(x,msg):
    if not x: raise RuntimeError(msg)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(p,v):
    t=Path(str(p)+'.tmp'); t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); t.replace(p)
def read_json(p): return json.loads(Path(p).read_text())
def read_gray(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unreadable image: {p}'); return a


## 2. Resolve and verify the Notebook 59R1 deployment bundle


In [ ]:
# ============================================================
# 2. Resolve and verify the Notebook 59R1 deployment bundle
# ============================================================

# Files physically stored inside <59R1 run>/bundle/
REQUIRED_BUNDLE_FILES = [
    'remove_model.joblib',
    'remove_calibrator.joblib',
    'predictor_schema.json',
    'policy.json',
    'reason_code_schema.json',
    'freeze_manifest.json',
    'runtime_contract.json',
]

# Files stored elsewhere in the 59R1 run.
REQUIRED_RUN_FILES = [
    'replay/replay_test.json',
]


def complete_59r1_runs(root):
    """
    Return completed Notebook 59R1 run directories.

    Expected structure:

      <run>/
        bundle/
          remove_model.joblib
          remove_calibrator.joblib
          predictor_schema.json
          policy.json
          reason_code_schema.json
          freeze_manifest.json
          runtime_contract.json

        replay/
          replay_test.json
    """

    root = Path(root)
    hits = []

    if not root.exists():
        return hits

    for run in root.iterdir():

        if not run.is_dir():
            continue

        bundle = run / 'bundle'

        if not bundle.is_dir():
            continue

        bundle_complete = all(
            (bundle / filename).exists()
            for filename in REQUIRED_BUNDLE_FILES
        )

        run_complete = all(
            (run / relative_path).exists()
            for relative_path in REQUIRED_RUN_FILES
        )

        if bundle_complete and run_complete:
            hits.append(run)

    return sorted(
        hits,
        key=lambda path: path.stat().st_mtime,
    )


# ------------------------------------------------------------
# Resolve one complete 59R1 run
# ------------------------------------------------------------

if NOTEBOOK59_RUN_OVERRIDE:

    N59 = Path(NOTEBOOK59_RUN_OVERRIDE)

    # Permit the override to point either at the run root
    # or directly at its bundle directory.
    if N59.name == 'bundle':
        N59 = N59.parent

    BUNDLE = N59 / 'bundle'
    REPLAY_DIR = N59 / 'replay'

    missing_bundle = [
        filename
        for filename in REQUIRED_BUNDLE_FILES
        if not (BUNDLE / filename).exists()
    ]

    missing_run = [
        relative_path
        for relative_path in REQUIRED_RUN_FILES
        if not (N59 / relative_path).exists()
    ]

    require(
        not missing_bundle and not missing_run,
        (
            f'Notebook 59R1 override is incomplete.\n'
            f'Run: {N59}\n'
            f'Missing bundle files: {missing_bundle}\n'
            f'Missing run files: {missing_run}'
        ),
    )

else:

    hits = complete_59r1_runs(N59_ROOT)

    require(
        hits,
        (
            'No complete Notebook 59R1 deployment run found '
            f'below {N59_ROOT}'
        ),
    )

    N59 = hits[-1]
    BUNDLE = N59 / 'bundle'
    REPLAY_DIR = N59 / 'replay'


# ------------------------------------------------------------
# Resolve all frozen control paths explicitly
# ------------------------------------------------------------

POLICY_PATH = BUNDLE / 'policy.json'
SCHEMA_PATH = BUNDLE / 'predictor_schema.json'
REASON_PATH = BUNDLE / 'reason_code_schema.json'
FREEZE_PATH = BUNDLE / 'freeze_manifest.json'
CONTRACT_PATH = BUNDLE / 'runtime_contract.json'

MODEL_PATH = BUNDLE / 'remove_model.joblib'
CALIBRATOR_PATH = BUNDLE / 'remove_calibrator.joblib'

REPLAY_PATH = REPLAY_DIR / 'replay_test.json'


# ------------------------------------------------------------
# Read control records
# ------------------------------------------------------------

policy = read_json(POLICY_PATH)
schema = read_json(SCHEMA_PATH)
reasons = read_json(REASON_PATH)
freeze = read_json(FREEZE_PATH)
contract = read_json(CONTRACT_PATH)
replay = read_json(REPLAY_PATH)


# ------------------------------------------------------------
# Core identity checks
# ------------------------------------------------------------

checks = {
    'bundle_directory_exists':
        BUNDLE.exists(),

    'replay_directory_exists':
        REPLAY_DIR.exists(),

    'deployment_bundle_complete':
        freeze.get('deployment_bundle_complete') is True,

    'replay_passed':
        replay.get('passed') is True,

    'policy_id_match':
        (
            policy.get('policy_id')
            == freeze.get('policy_id')
            == contract.get('policy_id')
            == replay.get('policy_id')
        ),

    'compatibility_id_match':
        (
            policy.get('compatibility_id')
            == freeze.get('compatibility_id')
            == contract.get('compatibility_id')
            == replay.get('compatibility_id')
        ),

    'safe_protect_disabled':
        (
            policy.get('safe_protect_enabled') is False
            and freeze.get('protect_included') is False
        ),

    'promotion_authority_false':
        (
            policy.get('promotion_authority') is False
            and freeze.get(
                'independent_promotion_authority'
            ) is False
        ),

    'schema_predictor_count_17':
        (
            schema.get('predictor_count') == 17
            and len(
                schema.get(
                    'ordered_predictors',
                    [],
                )
            ) == 17
        ),
}


# ------------------------------------------------------------
# Verify files sealed by freeze_manifest.json
# ------------------------------------------------------------

for item in freeze.get('objects', []):

    relative_path = item.get('relative_path')
    expected_hash = item.get('sha256')

    require(
        relative_path,
        '59R1 freeze manifest contains an object without relative_path.',
    )

    require(
        expected_hash,
        (
            '59R1 freeze manifest contains an object '
            f'without SHA-256: {relative_path}'
        ),
    )

    frozen_path = BUNDLE / relative_path

    checks[
        f'freeze_hash:{relative_path}'
    ] = (
        frozen_path.exists()
        and frozen_path.is_file()
        and sha256_file(frozen_path) == expected_hash
    )


# ------------------------------------------------------------
# Verify the primary runtime objects against policy.json
# ------------------------------------------------------------

checks['policy_model_hash'] = (
    MODEL_PATH.exists()
    and sha256_file(MODEL_PATH)
    == policy.get('model_sha256')
)

checks['policy_calibrator_hash'] = (
    CALIBRATOR_PATH.exists()
    and sha256_file(CALIBRATOR_PATH)
    == policy.get('calibrator_sha256')
)

checks['policy_schema_hash'] = (
    SCHEMA_PATH.exists()
    and sha256_file(SCHEMA_PATH)
    == policy.get('predictor_schema_sha256')
)

checks['policy_reason_schema_hash'] = (
    REASON_PATH.exists()
    and sha256_file(REASON_PATH)
    == policy.get('reason_code_schema_sha256')
)


# ------------------------------------------------------------
# Verify that policy filenames match the actual runtime files
# ------------------------------------------------------------

checks['model_filename_match'] = (
    policy.get('model_file')
    == MODEL_PATH.name
)

checks['calibrator_filename_match'] = (
    policy.get('calibrator_file')
    == CALIBRATOR_PATH.name
)

checks['schema_filename_match'] = (
    policy.get('predictor_schema_file')
    == SCHEMA_PATH.name
)

checks['reason_filename_match'] = (
    policy.get('reason_code_schema_file')
    == REASON_PATH.name
)


# ------------------------------------------------------------
# Explicit replay identity checks
# ------------------------------------------------------------

checks['replay_policy_match'] = (
    replay.get('policy_id')
    == policy.get('policy_id')
)

checks['replay_compatibility_match'] = (
    replay.get('compatibility_id')
    == policy.get('compatibility_id')
)

checks['replay_has_rows'] = (
    int(replay.get('rows', 0)) > 0
)

checks['replay_has_pages'] = (
    int(replay.get('pages', 0)) > 0
)

checks['replay_model_reload_exact'] = (
    replay.get('model_reload_exact') is True
)

checks['replay_calibrator_reload_exact'] = (
    replay.get('calibrator_reload_exact') is True
)


# ------------------------------------------------------------
# Write audit BEFORE failing
# ------------------------------------------------------------

bundle_integrity_record = {
    'notebook59_run':
        str(N59),

    'bundle':
        str(BUNDLE),

    'replay_directory':
        str(REPLAY_DIR),

    'replay_path':
        str(REPLAY_PATH),

    'policy_id':
        policy.get('policy_id'),

    'compatibility_id':
        policy.get('compatibility_id'),

    'checks':
        checks,
}

atomic_json(
    AUDIT / 'notebook59r1_bundle_integrity.json',
    bundle_integrity_record,
)


failed_checks = [
    name
    for name, passed in checks.items()
    if not bool(passed)
]

require(
    not failed_checks,
    (
        'Notebook 59R1 deployment bundle integrity failure.\n'
        f'Failed checks: {failed_checks}\n'
        'See notebook59r1_bundle_integrity.json.'
    ),
)


# ------------------------------------------------------------
# Load the frozen binary objects only AFTER integrity passes
# ------------------------------------------------------------

model = joblib.load(MODEL_PATH)
calibrator = joblib.load(CALIBRATOR_PATH)

predictors = schema.get('ordered_predictors')

require(
    isinstance(predictors, list)
    and len(predictors) == 17,
    (
        'Verified 59R1 schema does not contain '
        'exactly 17 ordered predictors.'
    ),
)


# ------------------------------------------------------------
# Basic executable-object checks
# ------------------------------------------------------------

require(
    hasattr(model, 'predict_proba'),
    (
        'Frozen REMOVE model does not expose '
        'predict_proba().'
    ),
)

require(
    hasattr(calibrator, 'predict'),
    (
        'Frozen REMOVE calibrator does not expose '
        'predict().'
    ),
)

if hasattr(model, 'n_features_in_'):

    require(
        int(model.n_features_in_)
        == len(predictors),
        (
            'Frozen REMOVE model feature count '
            f'{model.n_features_in_} does not match '
            f'the frozen schema count {len(predictors)}.'
        ),
    )


# ------------------------------------------------------------
# Record exact downstream source identity
# ------------------------------------------------------------

source_record = {
    'notebook59_run':
        str(N59),

    'bundle':
        str(BUNDLE),

    'replay_path':
        str(REPLAY_PATH),

    'policy_id':
        policy.get('policy_id'),

    'compatibility_id':
        policy.get('compatibility_id'),

    'model_path':
        str(MODEL_PATH),

    'model_sha256':
        sha256_file(MODEL_PATH),

    'calibrator_path':
        str(CALIBRATOR_PATH),

    'calibrator_sha256':
        sha256_file(CALIBRATOR_PATH),

    'predictor_schema_path':
        str(SCHEMA_PATH),

    'predictor_schema_sha256':
        sha256_file(SCHEMA_PATH),

    'freeze_manifest_sha256':
        sha256_file(FREEZE_PATH),

    'runtime_contract_sha256':
        sha256_file(CONTRACT_PATH),

    'replay_test_sha256':
        sha256_file(REPLAY_PATH),

    'predictor_count':
        len(predictors),

    'bundle_integrity_passed':
        True,
}

atomic_json(
    AUDIT / 'notebook59r1_source.json',
    source_record,
)


print('Notebook 59R1 run:')
print(N59)

print('\nBundle:')
print(BUNDLE)

print('\nReplay:')
print(REPLAY_PATH)

print('\nPolicy ID:')
print(policy.get('policy_id'))

print('\nCompatibility ID:')
print(policy.get('compatibility_id'))

print('\nPredictors:')
print(len(predictors))

print('\n59R1 deployment bundle integrity: PASS')

## 3. Canonical V7.8/M3 preparation and 17 frozen predictors


In [ ]:
MASK_DILATION=2; CROP_MARGIN=48; DARK_THRESHOLD=200
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]; require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask drift'); return out
def prepare_mask(seed): return cv2.dilate((seed>0).astype(np.uint8)*255,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
def telea(observed,mask): return immutable(observed,cv2.inpaint(observed,mask,3,cv2.INPAINT_TELEA),mask)
def background(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); num=cv2.GaussianBlur(observed.astype(np.float32)*visible,(0,0),sigma); den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma); fallback=cv2.GaussianBlur(observed,(0,0),sigma); return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
def support_maps(observed,mask,band=4):
    dark=(observed<190)&(mask==0); h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0; v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0; n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*band+1,2*band+1),np.uint8); return *(cv2.dilate(z.astype(np.uint8),k)>0 for z in [h,v,c]),
def geometry(observed,mask,t):
    paper=background(observed,mask); norm=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0; preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=80)&(norm>=80); ambiguous=m&~preserve&~remove; result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=t[ambiguous]; result[preserve]=observed[preserve]; return immutable(observed,result,mask),{'paper':paper,'norm':norm,'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact}
def frac(a,r=3): return cv2.boxFilter(a.astype(np.float32),-1,(2*r+1,2*r+1),normalize=True,borderType=cv2.BORDER_REFLECT)
def distance_to(a): return cv2.distanceTransform((~a).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32) if a.any() else np.full(a.shape,float(max(a.shape)),np.float32)
def predictor_frame(observed,m3,routes,p_print):
    edge=cv2.Canny(observed,70,170)>0; boundary=m3^(cv2.erode(m3.astype(np.uint8),np.ones((3,3),np.uint8))>0); protected=routes['preserve']; _,components=cv2.connectedComponents(m3.astype(np.uint8),8); ys,xs=np.where(m3)
    d={'observed_value':observed,'normalised_value':routes['norm'],'paper_value':routes['paper'],'mask_depth':cv2.distanceTransform(m3.astype(np.uint8),cv2.DIST_L2,5),'boundary_distance':distance_to(boundary),'distance_to_protected':distance_to(protected),'edge':edge,'local_edge_fraction':frac(edge),'local_dark_fraction':frac(observed<190),'local_preserve_fraction':frac(routes['preserve']),'local_remove_fraction':frac(routes['remove']),'local_ambiguous_fraction':frac(routes['ambiguous']),'local_horizontal_fraction':frac(routes['horizontal']),'local_vertical_fraction':frac(routes['vertical']),'local_compact_fraction':frac(routes['compact']),'p_print':p_print,'component_id':components}
    frame=pd.DataFrame({'y':ys,'x':xs,**{k:np.asarray(v)[ys,xs] for k,v in d.items()}}); require(list(frame[predictors].columns)==predictors,'Predictor order mismatch'); X=frame[predictors].apply(pd.to_numeric,errors='coerce'); require(not np.isinf(X.to_numpy(float)).any(),'Infinite predictor'); return frame,X


## 4. Resolve Notebook 06 -> 07 reviewed HITL cohort


In [ ]:
# ============================================================
# CODE CELL 4
# Resolve the canonical Notebook 07 reviewed HITL cohort
# ============================================================
#
# Notebook 07 is authoritative for:
#   - the aligned observed page;
#   - the human-amended/corrected annotation mask;
#   - the canonical reviewed page identity.
#
# Notebook 06 draft masks and confidence maps are not required
# for admission to this cohort.
#
# This cell does not require p_print. That is resolved
# separately in Cell 5 because it belongs to the observed-print
# evidence lineage, not the annotation-mask lineage.
# ============================================================


def load_json_object(path):

    value = read_json(path)

    require(
        isinstance(value, dict),
        f'JSON root is not an object: {path}',
    )

    return value


def first_existing_path(
    record,
    field_names,
):

    for field_name in field_names:

        value = record.get(
            field_name
        )

        if value is None:
            continue

        value = str(value).strip()

        if not value:
            continue

        path = Path(value)

        if (
            path.exists()
            and path.is_file()
        ):
            return path, field_name

    return None, None


def canonical_page_name(
    review,
    observed_path,
    mask_path,
    audit_path,
):

    # Prefer an explicit page identity if recorded.
    for field_name in [
        'page',
        'page_id',
        'page_name',
        'canonical_page',
    ]:

        value = review.get(
            field_name
        )

        if value is not None:

            value = str(value).strip()

            if value:
                return value

    # Otherwise use the aligned observed output stem.
    if observed_path is not None:
        return observed_path.stem

    # Then use the reviewed-mask output stem.
    if mask_path is not None:
        return mask_path.stem

    # Final diagnostic fallback.
    return audit_path.stem


# ------------------------------------------------------------
# 1. Read every Notebook 07 review audit
# ------------------------------------------------------------

review_records = []
review_parse_failures = []


for audit_path in sorted(
    Path(
        REVIEW_AUDIT_DIR
    ).glob('*.json')
):

    try:

        review = load_json_object(
            audit_path
        )

        review['_audit_path'] = str(
            audit_path
        )

        review_records.append(
            review
        )

    except Exception as error:

        review_parse_failures.append({
            'review_audit_path':
                str(audit_path),

            'error':
                str(error),
        })


pd.DataFrame(
    review_parse_failures,
    columns=[
        'review_audit_path',
        'error',
    ],
).to_csv(
    AUDIT
    / 'notebook07_review_audit_parse_failures.csv',
    index=False,
)


require(
    len(review_records) > 0,
    (
        'No readable Notebook 07 review audits '
        f'were found in {REVIEW_AUDIT_DIR}.'
    ),
)


require(
    not review_parse_failures,
    (
        f'{len(review_parse_failures)} Notebook 07 '
        'review audit records could not be read. '
        'See notebook07_review_audit_parse_failures.csv.'
    ),
)


# ------------------------------------------------------------
# 2. Resolve the authoritative observed page and reviewed mask
# ------------------------------------------------------------

resolved_rows = []
unresolved_rows = []


for review in review_records:

    audit_path = Path(
        review['_audit_path']
    )

    observed_path, observed_field = (
        first_existing_path(
            review,
            [
                'aligned_output',
                'aligned_page',
                'observed_output',
                'aligned_image',
                'observed_path',
            ],
        )
    )

    reviewed_mask_path, mask_field = (
        first_existing_path(
            review,
            [
                'corrected_mask_output',
                'corrected_mask',
                'reviewed_mask_output',
                'reviewed_mask',
                'mask_output',
            ],
        )
    )

    page = canonical_page_name(
        review,
        observed_path,
        reviewed_mask_path,
        audit_path,
    )


    if observed_path is None:

        unresolved_rows.append({
            'page':
                page,

            'review_audit_path':
                str(audit_path),

            'reason':
                'aligned_observed_page_not_resolved',
        })

        continue


    if reviewed_mask_path is None:

        unresolved_rows.append({
            'page':
                page,

            'review_audit_path':
                str(audit_path),

            'reason':
                'reviewed_mask_not_resolved',
        })

        continue


    try:

        observed = read_gray(
            observed_path
        )

        reviewed_mask = read_gray(
            reviewed_mask_path
        )

        require(
            observed.ndim == 2,
            (
                f'{page}: aligned observed page '
                'is not two-dimensional.'
            ),
        )

        require(
            reviewed_mask.ndim == 2,
            (
                f'{page}: reviewed mask '
                'is not two-dimensional.'
            ),
        )

        require(
            observed.shape
            == reviewed_mask.shape,
            (
                f'{page}: aligned observed '
                'and reviewed-mask shapes differ: '
                f'{observed.shape} versus '
                f'{reviewed_mask.shape}.'
            ),
        )

        reviewed_mask_binary = (
            reviewed_mask > 0
        )

        require(
            reviewed_mask_binary.any(),
            (
                f'{page}: reviewed mask '
                'contains no positive pixels.'
            ),
        )

    except Exception as error:

        unresolved_rows.append({
            'page':
                page,

            'review_audit_path':
                str(audit_path),

            'observed_path':
                str(observed_path),

            'reviewed_mask_path':
                str(reviewed_mask_path),

            'reason':
                'invalid_reviewed_page_pair',

            'error':
                str(error),
        })

        continue


    resolved_rows.append({
        'page':
            page,

        'observed_path':
            str(observed_path),

        'reviewed_mask_path':
            str(reviewed_mask_path),

        'review_audit_path':
            str(audit_path),

        'observed_field':
            observed_field,

        'reviewed_mask_field':
            mask_field,

        'observed_sha256':
            sha256_file(
                observed_path
            ),

        'reviewed_mask_sha256':
            sha256_file(
                reviewed_mask_path
            ),

        'height':
            int(
                observed.shape[0]
            ),

        'width':
            int(
                observed.shape[1]
            ),

        'reviewed_mask_pixels':
            int(
                reviewed_mask_binary.sum()
            ),

        'mask_authority':
            'NOTEBOOK07_HUMAN_REVIEWED',

        'notebook06_required':
            False,

        'yolo_confidence_used':
            False,
    })


# ------------------------------------------------------------
# 3. Construct the canonical reviewed-page roster
# ------------------------------------------------------------

roster = pd.DataFrame(
    resolved_rows
)


pd.DataFrame(
    unresolved_rows
).to_csv(
    TABLES
    / 'unresolved_notebook07_reviews.csv',
    index=False,
)


require(
    not roster.empty,
    (
        'No valid Notebook 07 reviewed pages '
        'could be resolved. '
        'Inspect unresolved_notebook07_reviews.csv.'
    ),
)


# Multiple audit records for the same page must not silently
# select different physical files.
duplicate_pages = (
    roster[
        roster.duplicated(
            'page',
            keep=False,
        )
    ]
    .sort_values(
        [
            'page',
            'review_audit_path',
        ]
    )
)


if len(duplicate_pages):

    duplicate_summary = (
        duplicate_pages
        .groupby(
            'page',
            as_index=False,
        )
        .agg(
            records=(
                'review_audit_path',
                'size',
            ),

            observed_identities=(
                'observed_sha256',
                'nunique',
            ),

            mask_identities=(
                'reviewed_mask_sha256',
                'nunique',
            ),
        )
    )

    conflicting_pages = (
        duplicate_summary[
            (
                duplicate_summary[
                    'observed_identities'
                ] != 1
            )
            |
            (
                duplicate_summary[
                    'mask_identities'
                ] != 1
            )
        ]
    )

    duplicate_pages.to_csv(
        AUDIT
        / 'duplicate_notebook07_page_records.csv',
        index=False,
    )

    require(
        conflicting_pages.empty,
        (
            'Conflicting Notebook 07 records exist '
            'for the same canonical page. '
            'See duplicate_notebook07_page_records.csv.'
        ),
    )

    # Identical audit replicas are collapsed deterministically.
    roster = (
        roster
        .sort_values(
            [
                'page',
                'review_audit_path',
            ]
        )
        .drop_duplicates(
            'page',
            keep='first',
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 4. Apply an optional explicit page allowlist
# ------------------------------------------------------------

if PAGE_ALLOWLIST is not None:

    requested_pages = {
        str(value)
        for value in PAGE_ALLOWLIST
    }

    available_pages = set(
        roster[
            'page'
        ].astype(str)
    )

    missing_allowlisted_pages = sorted(
        requested_pages
        - available_pages
    )

    require(
        not missing_allowlisted_pages,
        (
            'Allowlisted Notebook 07 pages '
            'could not be resolved: '
            f'{missing_allowlisted_pages}'
        ),
    )

    roster = (
        roster[
            roster[
                'page'
            ].astype(str).isin(
                requested_pages
            )
        ]
        .copy()
    )


roster = (
    roster
    .sort_values('page')
    .reset_index(drop=True)
)


if REQUIRE_EXACT_151:

    require(
        len(roster)
        == EXPECTED_REVIEWED_PAGES,
        (
            'Unexpected Notebook 07 reviewed-page count: '
            f'expected={EXPECTED_REVIEWED_PAGES}, '
            f'resolved={len(roster)}.'
        ),
    )


# ------------------------------------------------------------
# 5. Persist and display the resolved cohort
# ------------------------------------------------------------

roster.to_csv(
    TABLES
    / 'hitl_reviewed_roster.csv',
    index=False,
)


resolution_summary = {
    'review_audit_directory':
        str(
            REVIEW_AUDIT_DIR
        ),

    'review_audits_read':
        len(
            review_records
        ),

    'reviewed_pages_resolved':
        len(
            roster
        ),

    'review_records_unresolved':
        len(
            unresolved_rows
        ),

    'expected_reviewed_pages':
        EXPECTED_REVIEWED_PAGES,

    'exact_count_required':
        REQUIRE_EXACT_151,

    'mask_authority':
        'Notebook 07 human-amended corrected mask',

    'notebook06_required':
        False,

    'yolo_confidence_used':
        False,

    'p_print_resolved_in_this_cell':
        False,
}


atomic_json(
    AUDIT
    / 'hitl_reviewed_roster_summary.json',
    resolution_summary,
)


print(
    json.dumps(
        resolution_summary,
        indent=2,
    )
)


display(
    roster[
        [
            'page',
            'observed_path',
            'reviewed_mask_path',
            'height',
            'width',
            'reviewed_mask_pixels',
        ]
    ]
)

## 5. Frozen REMOVE scoring, M5 reconstruction and fail-closed pages


In [ ]:
# ============================================================
# CODE CELL 5
# Build canonical M3/V7.8 arrays and the exact 59R1 predictors
# ============================================================
#
# Required inputs:
#
#   Notebook 07 aligned observed page
#   Notebook 07 human-reviewed corrected mask
#   genuine observed-print p_print array
#
# Prohibited substitutions:
#
#   Notebook 06 YOLO confidence != p_print
#   reviewed mask values       != p_print
#   V7.8 preserve route        != p_print
#
# If no genuine p_print array is available, this cell stops
# before the 59R1 REMOVE model is scored.
# ============================================================


# ------------------------------------------------------------
# 1. Configuration for genuine observed-print probabilities
# ------------------------------------------------------------

# Place one floating-point .npy array per reviewed page here.
#
# Required naming:
#
#     <canonical Notebook 07 page>.npy
#
# Example:
#
#     Page001.npy
#     Page002.npy
#
# Each array must:
#
#   - have exactly the same height/width as the observed page;
#   - contain finite values;
#   - have values in [0, 1];
#   - represent calibrated observed-print probability;
#   - not be the Notebook 06 YOLO confidence map.

P_PRINT_ROOT = (
    ROOT
    / 'observed_print_runtime'
    / 'p_print'
)


# Leave False.
# This exists only to make the safety boundary explicit.
ALLOW_MISSING_P_PRINT = False


# ------------------------------------------------------------
# 2. Canonical frozen V7.8 preparation functions
# ------------------------------------------------------------

MASK_DILATION = 2
CROP_MARGIN = 48
GEOMETRY_THRESHOLD = 80
BAND_WIDTH = 4


def binary_mask(mask):

    return np.where(
        mask > 0,
        255,
        0,
    ).astype(
        np.uint8
    )


def prepare_mask(
    seed,
    dilation=MASK_DILATION,
):

    mask = binary_mask(
        seed
    )

    if dilation > 0:

        kernel = (
            cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (
                    2 * dilation + 1,
                    2 * dilation + 1,
                ),
            )
        )

        mask = cv2.dilate(
            mask,
            kernel,
        )

    return mask


def immutable(
    original,
    candidate,
    mask,
):

    output = original.copy()

    output[
        mask > 0
    ] = candidate[
        mask > 0
    ]

    require(
        np.array_equal(
            output[
                mask == 0
            ],
            original[
                mask == 0
            ],
        ),
        'Candidate changed a pixel outside the authoritative mask.',
    )

    return output


def padded_components(
    mask,
):

    number,
    labels,
    statistics,
    _ = cv2.connectedComponentsWithStats(
        (
            mask > 0
        ).astype(
            np.uint8
        ),
        8,
    )

    height, width = mask.shape

    boxes = []

    for component_id in range(
        1,
        number,
    ):

        x, y, box_width, box_height, area = (
            statistics[
                component_id
            ]
        )

        if area < 8:
            continue

        boxes.append((
            max(
                0,
                int(x) - CROP_MARGIN,
            ),
            max(
                0,
                int(y) - CROP_MARGIN,
            ),
            min(
                width,
                int(x + box_width)
                + CROP_MARGIN,
            ),
            min(
                height,
                int(y + box_height)
                + CROP_MARGIN,
            ),
        ))

    changed = True

    while changed:

        changed = False
        merged = []

        while boxes:

            (
                ax0,
                ay0,
                ax1,
                ay1,
            ) = boxes.pop()

            remaining = []

            for (
                bx0,
                by0,
                bx1,
                by1,
            ) in boxes:

                overlaps = not (
                    ax1 < bx0
                    or bx1 < ax0
                    or ay1 < by0
                    or by1 < ay0
                )

                if overlaps:

                    ax0 = min(
                        ax0,
                        bx0,
                    )

                    ay0 = min(
                        ay0,
                        by0,
                    )

                    ax1 = max(
                        ax1,
                        bx1,
                    )

                    ay1 = max(
                        ay1,
                        by1,
                    )

                    changed = True

                else:

                    remaining.append((
                        bx0,
                        by0,
                        bx1,
                        by1,
                    ))

            boxes = remaining

            merged.append((
                ax0,
                ay0,
                ax1,
                ay1,
            ))

        boxes = merged

    return boxes


def telea_reconstruct(
    observed,
    mask,
):

    result = observed.copy()

    for (
        x0,
        y0,
        x1,
        y1,
    ) in padded_components(
        mask
    ):

        observed_crop = observed[
            y0:y1,
            x0:x1,
        ]

        mask_crop = mask[
            y0:y1,
            x0:x1,
        ]

        candidate = cv2.inpaint(
            observed_crop,
            mask_crop,
            3,
            cv2.INPAINT_TELEA,
        )

        local = result[
            y0:y1,
            x0:x1,
        ]

        local[
            mask_crop > 0
        ] = candidate[
            mask_crop > 0
        ]

        result[
            y0:y1,
            x0:x1,
        ] = local

    return immutable(
        observed,
        result,
        mask,
    )


def background_surface(
    observed,
    mask,
    sigma=45.0,
):

    visible = (
        (mask == 0)
        & (observed >= 200)
    )

    numerator = cv2.GaussianBlur(
        observed.astype(
            np.float32
        )
        * visible,
        (0, 0),
        sigma,
    )

    denominator = cv2.GaussianBlur(
        visible.astype(
            np.float32
        ),
        (0, 0),
        sigma,
    )

    fallback = cv2.GaussianBlur(
        observed,
        (0, 0),
        sigma,
    )

    surface = (
        numerator
        / np.maximum(
            denominator,
            1e-4,
        )
    )

    return np.clip(
        np.where(
            denominator > 0.02,
            surface,
            fallback,
        ),
        0,
        255,
    ).astype(
        np.uint8
    )


def support_maps(
    observed,
    mask,
):

    dark = (
        (observed < 190)
        & (mask == 0)
    )

    horizontal = (
        cv2.morphologyEx(
            dark.astype(
                np.uint8
            )
            * 255,
            cv2.MORPH_OPEN,
            np.ones(
                (1, 35),
                np.uint8,
            ),
        )
        > 0
    )

    vertical = (
        cv2.morphologyEx(
            dark.astype(
                np.uint8
            )
            * 255,
            cv2.MORPH_OPEN,
            np.ones(
                (15, 1),
                np.uint8,
            ),
        )
        > 0
    )

    (
        number,
        labels,
        statistics,
        _,
    ) = cv2.connectedComponentsWithStats(
        dark.astype(
            np.uint8
        ),
        8,
    )

    compact = np.zeros_like(
        dark
    )

    for component_id in range(
        1,
        number,
    ):

        (
            x,
            y,
            width,
            height,
            area,
        ) = statistics[
            component_id
        ]

        if (
            3 <= area <= 650
            and width <= 40
            and height <= 40
        ):

            compact[
                labels
                == component_id
            ] = True

    kernel = np.ones(
        (
            2 * BAND_WIDTH + 1,
            2 * BAND_WIDTH + 1,
        ),
        np.uint8,
    )

    return (
        cv2.dilate(
            horizontal.astype(
                np.uint8
            ),
            kernel,
        )
        > 0,

        cv2.dilate(
            vertical.astype(
                np.uint8
            ),
            kernel,
        )
        > 0,

        cv2.dilate(
            compact.astype(
                np.uint8
            ),
            kernel,
        )
        > 0,
    )


def geometry_t80(
    observed,
    mask,
    telea,
):

    paper = background_surface(
        observed,
        mask,
    )

    normalised = cv2.divide(
        observed,
        np.maximum(
            paper,
            1,
        ),
        scale=255,
    )

    (
        horizontal,
        vertical,
        compact,
    ) = support_maps(
        observed,
        mask,
    )

    inside = (
        mask > 0
    )

    preserve = (
        inside
        & (
            horizontal
            | vertical
            | compact
        )
        & (
            observed < 200
        )
    )

    remove = (
        inside
        & ~preserve
        & (
            observed
            >= GEOMETRY_THRESHOLD
        )
        & (
            normalised
            >= GEOMETRY_THRESHOLD
        )
    )

    ambiguous = (
        inside
        & ~preserve
        & ~remove
    )

    result = observed.copy()

    result[
        remove
    ] = paper[
        remove
    ]

    result[
        ambiguous
    ] = telea[
        ambiguous
    ]

    result[
        preserve
    ] = observed[
        preserve
    ]

    result = immutable(
        observed,
        result,
        mask,
    )

    routes = {
        'paper':
            paper,

        'normalised':
            normalised,

        'preserve':
            preserve,

        'remove':
            remove,

        'ambiguous':
            ambiguous,

        'horizontal':
            horizontal,

        'vertical':
            vertical,

        'compact':
            compact,
    }

    return result, routes


# ------------------------------------------------------------
# 3. Local predictor helpers
# ------------------------------------------------------------

def local_fraction(
    mask,
    radius=3,
):

    size = (
        2 * radius + 1
    )

    return cv2.boxFilter(
        mask.astype(
            np.float32
        ),
        -1,
        (
            size,
            size,
        ),
        normalize=True,
        borderType=cv2.BORDER_REFLECT,
    )


def distance_to_true(
    mask,
):

    mask = mask.astype(
        bool
    )

    if not mask.any():

        return np.full(
            mask.shape,
            float(
                max(
                    mask.shape
                )
            ),
            dtype=np.float32,
        )

    return cv2.distanceTransform(
        (
            ~mask
        ).astype(
            np.uint8
        ),
        cv2.DIST_L2,
        5,
    ).astype(
        np.float32
    )


# ------------------------------------------------------------
# 4. Genuine p_print resolver
# ------------------------------------------------------------

def resolve_p_print_path(
    page,
):

    candidates = [
        P_PRINT_ROOT
        / f'{page}.npy',

        P_PRINT_ROOT
        / f'{page}_p_print.npy',

        P_PRINT_ROOT
        / page
        / 'p_print.npy',
    ]

    matches = [
        path
        for path in candidates
        if (
            path.exists()
            and path.is_file()
        )
    ]

    # Remove paths that resolve to the same physical file.
    unique = {}

    for path in matches:

        unique[
            str(
                path.resolve()
            )
        ] = path

    matches = list(
        unique.values()
    )

    require(
        len(matches) == 1,
        (
            f'{page}: expected exactly one genuine '
            'observed-print p_print array beneath '
            f'{P_PRINT_ROOT}; found {len(matches)}. '
            'Notebook 06 YOLO confidence must not '
            'be substituted.'
        ),
    )

    return matches[0]


def load_p_print(
    page,
    expected_shape,
):

    path = resolve_p_print_path(
        page
    )

    values = np.load(
        path
    )

    require(
        values.ndim == 2,
        (
            f'{page}: p_print array '
            'is not two-dimensional.'
        ),
    )

    require(
        values.shape
        == expected_shape,
        (
            f'{page}: p_print shape '
            f'{values.shape} does not match '
            f'observed-page shape '
            f'{expected_shape}.'
        ),
    )

    values = values.astype(
        np.float32
    )

    require(
        np.isfinite(
            values
        ).all(),
        (
            f'{page}: p_print contains '
            'non-finite values.'
        ),
    )

    require(
        (
            values >= 0
        ).all()
        and (
            values <= 1
        ).all(),
        (
            f'{page}: p_print values '
            'are outside [0, 1].'
        ),
    )

    return values, path


# ------------------------------------------------------------
# 5. Build the exact ordered 59R1 predictor frame
# ------------------------------------------------------------

def build_remove_predictors(
    page,
    observed,
    prepared_mask,
    routes,
    p_print,
):

    inside = (
        prepared_mask > 0
    )

    require(
        p_print.shape
        == observed.shape,
        (
            f'{page}: p_print/observed '
            'shape mismatch.'
        ),
    )

    edge = (
        cv2.Canny(
            observed,
            70,
            170,
        )
        > 0
    )

    mask_depth = (
        cv2.distanceTransform(
            inside.astype(
                np.uint8
            ),
            cv2.DIST_L2,
            5,
        ).astype(
            np.float32
        )
    )

    eroded = (
        cv2.erode(
            inside.astype(
                np.uint8
            ),
            np.ones(
                (3, 3),
                np.uint8,
            ),
        )
        > 0
    )

    boundary = (
        inside
        ^ eroded
    )

    boundary_distance = (
        distance_to_true(
            boundary
        )
    )

    distance_to_protected = (
        distance_to_true(
            routes[
                'preserve'
            ]
        )
    )

    (
        _,
        component_ids,
    ) = cv2.connectedComponents(
        inside.astype(
            np.uint8
        ),
        connectivity=8,
    )

    y_values, x_values = np.where(
        inside
    )

    frame = pd.DataFrame({
        'page':
            page,

        'y':
            y_values,

        'x':
            x_values,

        'observed_value':
            observed[
                y_values,
                x_values,
            ].astype(
                np.float32
            ),

        'normalised_value':
            routes[
                'normalised'
            ][
                y_values,
                x_values,
            ].astype(
                np.float32
            ),

        'paper_value':
            routes[
                'paper'
            ][
                y_values,
                x_values,
            ].astype(
                np.float32
            ),

        'mask_depth':
            mask_depth[
                y_values,
                x_values,
            ],

        'boundary_distance':
            boundary_distance[
                y_values,
                x_values,
            ],

        'distance_to_protected':
            distance_to_protected[
                y_values,
                x_values,
            ],

        'edge':
            edge[
                y_values,
                x_values,
            ].astype(
                np.float32
            ),

        'local_edge_fraction':
            local_fraction(
                edge
            )[
                y_values,
                x_values,
            ],

        'local_dark_fraction':
            local_fraction(
                observed < 190
            )[
                y_values,
                x_values,
            ],

        'local_preserve_fraction':
            local_fraction(
                routes[
                    'preserve'
                ]
            )[
                y_values,
                x_values,
            ],

        'local_remove_fraction':
            local_fraction(
                routes[
                    'remove'
                ]
            )[
                y_values,
                x_values,
            ],

        'local_ambiguous_fraction':
            local_fraction(
                routes[
                    'ambiguous'
                ]
            )[
                y_values,
                x_values,
            ],

        'local_horizontal_fraction':
            local_fraction(
                routes[
                    'horizontal'
                ]
            )[
                y_values,
                x_values,
            ],

        'local_vertical_fraction':
            local_fraction(
                routes[
                    'vertical'
                ]
            )[
                y_values,
                x_values,
            ],

        'local_compact_fraction':
            local_fraction(
                routes[
                    'compact'
                ]
            )[
                y_values,
                x_values,
            ],

        'p_print':
            p_print[
                y_values,
                x_values,
            ],

        'component_id':
            component_ids[
                y_values,
                x_values,
            ].astype(
                np.float32
            ),
    })


    missing_predictors = [
        predictor
        for predictor in predictors
        if predictor not in frame.columns
    ]

    require(
        not missing_predictors,
        (
            f'{page}: missing frozen '
            f'59R1 predictors: '
            f'{missing_predictors}'
        ),
    )


    X = frame.loc[
        :,
        predictors,
    ].copy()


    require(
        list(
            X.columns
        )
        == predictors,
        (
            f'{page}: predictor order '
            'does not match the frozen '
            '59R1 schema.'
        ),
    )


    for predictor in predictors:

        X[
            predictor
        ] = pd.to_numeric(
            X[
                predictor
            ],
            errors='coerce',
        )


    require(
        not np.isinf(
            X.to_numpy(
                dtype=np.float64
            )
        ).any(),
        (
            f'{page}: infinite predictor '
            'value found.'
        ),
    )


    return frame, X


# ------------------------------------------------------------
# 6. Preflight the complete reviewed cohort
#
# This deliberately stops before any 59R1 scoring if genuine
# p_print is unavailable for one or more pages.
# ------------------------------------------------------------

preflight_rows = []
preflight_failures = []


for row in roster.itertuples(
    index=False
):

    page = str(
        row.page
    )

    try:

        observed = read_gray(
            Path(
                row.observed_path
            )
        )

        reviewed_seed = read_gray(
            Path(
                row.reviewed_mask_path
            )
        )

        require(
            observed.shape
            == reviewed_seed.shape,
            (
                f'{page}: Notebook 07 '
                'observed/mask shape mismatch.'
            ),
        )

        prepared_mask = prepare_mask(
            reviewed_seed,
            MASK_DILATION,
        )

        telea = telea_reconstruct(
            observed,
            prepared_mask,
        )

        baseline, routes = geometry_t80(
            observed,
            prepared_mask,
            telea,
        )

        p_print, p_print_path = (
            load_p_print(
                page,
                observed.shape,
            )
        )

        frame, X = (
            build_remove_predictors(
                page,
                observed,
                prepared_mask,
                routes,
                p_print,
            )
        )

        preflight_rows.append({
            'page':
                page,

            'status':
                'PASS',

            'observed_path':
                row.observed_path,

            'reviewed_mask_path':
                row.reviewed_mask_path,

            'p_print_path':
                str(
                    p_print_path
                ),

            'p_print_sha256':
                sha256_file(
                    p_print_path
                ),

            'observed_shape':
                str(
                    observed.shape
                ),

            'reviewed_seed_pixels':
                int(
                    (
                        reviewed_seed > 0
                    ).sum()
                ),

            'prepared_m3_pixels':
                int(
                    (
                        prepared_mask > 0
                    ).sum()
                ),

            'predictor_rows':
                len(
                    frame
                ),

            'predictor_columns':
                len(
                    X.columns
                ),

            'baseline_outside_mask_exact':
                bool(
                    np.array_equal(
                        baseline[
                            prepared_mask == 0
                        ],
                        observed[
                            prepared_mask == 0
                        ],
                    )
                ),
        })

    except Exception as error:

        preflight_failures.append({
            'page':
                page,

            'status':
                'FAIL_CLOSED',

            'error':
                str(error),
        })


preflight_df = pd.DataFrame(
    preflight_rows
)

preflight_failure_df = pd.DataFrame(
    preflight_failures
)


preflight_df.to_csv(
    TABLES
    / 'fresh_remove_predictor_preflight.csv',
    index=False,
)

preflight_failure_df.to_csv(
    TABLES
    / 'fresh_remove_predictor_preflight_failures.csv',
    index=False,
)


preflight_summary = {
    'reviewed_pages':
        len(
            roster
        ),

    'preflight_passed':
        len(
            preflight_df
        ),

    'preflight_failed_closed':
        len(
            preflight_failure_df
        ),

    'p_print_root':
        str(
            P_PRINT_ROOT
        ),

    'notebook06_confidence_used':
        False,

    'reviewed_mask_is_authority':
        True,

    'ready_for_59r1_remove_scoring':
        bool(
            len(
                preflight_failure_df
            )
            == 0
            and len(
                preflight_df
            )
            == len(
                roster
            )
        ),
}


atomic_json(
    AUDIT
    / 'fresh_remove_predictor_preflight_summary.json',
    preflight_summary,
)


print(
    json.dumps(
        preflight_summary,
        indent=2,
    )
)


if len(
    preflight_failure_df
):

    display(
        preflight_failure_df
    )


require(
    len(
        preflight_failure_df
    )
    == 0,
    (
        'Fresh 59R1 REMOVE scoring is blocked because '
        f'{len(preflight_failure_df)} reviewed page(s) '
        'lack a valid genuine p_print array or failed '
        'predictor preflight. See '
        'fresh_remove_predictor_preflight_failures.csv. '
        'Do not substitute Notebook 06 YOLO confidence.'
    ),
)


print(
    'Notebook 07 cohort and all 17 frozen predictors: PASS'
)

## 6. Replay, verdict, visual queue and final upload package


In [ ]:
require(len(page_df)>0,'No page passed')
sample=page_df.iloc[0].page; rr=roster.set_index('page').loc[sample]; observed=read_gray(rr.observed_path); m3=read_gray(PAGES/sample/'m3.png')>0; safe=read_gray(PAGES/sample/'safe_remove.png')>0; m5=read_gray(PAGES/sample/'m5_remove.png')>0; saved=read_gray(PAGES/sample/'corrected.png'); replayed=telea(observed,m5.astype(np.uint8)*255); replayed[safe]=observed[safe]
replay_record={'page':sample,'m5_formula_exact':bool(np.array_equal(m5,m3&~safe)),'corrected_exact':bool(np.array_equal(saved,replayed))}; atomic_json(AUDIT/'deterministic_replay.json',replay_record)
checks2={'59r1_bundle_verified':True,'all_pages_accounted':len(page_df)+len(fail_df)==len(roster),'zero_page_failures':len(fail_df)==0,'zero_outside_drift':bool((page_df.outside_authority_drift==0).all()),'zero_m5_xor':bool((page_df.m5_formula_xor==0).all()),'replay_exact':all(replay_record.values()) if False else replay_record['m5_formula_exact'] and replay_record['corrected_exact'],'no_clean_truth':True,'safe_protect_disabled':True,'no_refit':True}
first=next((k for k,v in checks2.items() if not v),None); verdict='PASS_60R2_V9_0B_FRESH_SAFE_REMOVE' if first is None else 'FAIL_OR_PARTIAL_60R2_V9_0B'
record={'version':'60R2-v9.0B-fresh-safe-remove-HITL-59R1','verdict':verdict,'first_failed_gate':first,'checks':checks2,'resolved_pages':len(roster),'passed_pages':len(page_df),'failed_pages':len(fail_df),'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'safe_remove_total':int(page_df.safe_remove_pixels.sum()),'m3_total':int(page_df.m3_pixels.sum()),'m5_total':int(page_df.m5_pixels.sum()),'production_authority':False}; atomic_json(OUT/'notebook60r2_verdict.json',record)
queue=page_df.copy(); queue['safe_remove_fraction']=queue.safe_remove_pixels/np.maximum(queue.m3_pixels,1); queue.sort_values(['safe_remove_fraction','safe_remove_pixels'],ascending=False).to_csv(TABLES/'visual_review_queue.csv',index=False)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'run':str(OUT),'version':record['version'],'bundle':str(BUNDLE),'policy_sha256':sha256_file(BUNDLE/'policy.json'),'schema_sha256':sha256_file(BUNDLE/'predictor_schema.json'),'model_sha256':sha256_file(BUNDLE/policy['model_file']),'calibrator_sha256':sha256_file(BUNDLE/policy['calibrator_file']),'roster_sha256':sha256_file(TABLES/'hitl_reviewed_roster.csv'),'environment':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}; atomic_json(OUT/'experiment_identity.json',identity)
lines=['NOTEBOOK 60R2 / V9.0B FRESH SAFE_REMOVE HITL OPERATIONALISATION','',f'Verdict: {verdict}',f'Pages passed / failed: {len(page_df)} / {len(fail_df)}',f'59R1 policy: {policy["policy_id"]}',f'SAFE_REMOVE / M3 / M5: {record["safe_remove_total"]:,} / {record["m3_total"]:,} / {record["m5_total"]:,}','','59R1 joblib objects loaded from Drive and hash-verified.','No clean truth, refit, recalibration, SAFE_PROTECT or production authority.']; (OUT/'executive_summary.txt').write_text('
'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook60r2_verdict.json',OUT/'experiment_identity.json',AUDIT/'notebook59r1_source.json',AUDIT/'deterministic_replay.json',TABLES/'hitl_reviewed_roster.csv',TABLES/'page_results.csv',TABLES/'page_failures.csv',TABLES/'visual_review_queue.csv']
if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists():required.append(TABLES/'all_fresh_safe_remove_predictions.parquet')
rows=[]
for src in required:
    require(src.exists(),f'Missing upload artifact {src}'); cat='tables' if src.parent==TABLES else 'audit' if src.parent==AUDIT else 'root'; dst=UPLOAD/cat/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'README.txt').write_text('60R2 uses the hash-verified Notebook 59R1 Drive bundle. Binary joblib files are intentionally not included. Research/HITL evidence only.
'); zip_path=UPLOAD/'60R2_V9_0B_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('
'.join(lines)); print('Upload bundle:',zip_path)
